<a href="https://colab.research.google.com/github/puneethhp446/multilingual-courtroom-judgment-simplifier-/blob/main/notebooks/01_phase1_and_2_cleaning_and_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import pandas as pd
from sklearn.model_selection import train_test_split

BASE = '/content/drive/MyDrive/judgment-simplifier'
RAW = f'{BASE}/data/raw/synthetic_indian_civil_judgments_2000.csv'
for d in ['data/processed', 'outputs', 'models', 'notebooks']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

print('CSV found:', os.path.exists(RAW))

CLEAN = f'{BASE}/data/processed/clean.csv'
if os.path.exists(CLEAN):
    df = pd.read_csv(CLEAN)
    print('Loaded existing clean.csv')
else:
    df = pd.read_csv(RAW)
    df = df.dropna().drop_duplicates()
    for c in df.columns:
        df[c] = df[c].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

    hdr = df['Facts_of_the_Case'].str.extract(r'^\[(.*?)\|(.*?)\|(.*?)\]\s*(.*)$', flags=re.S)
    df['Court'] = hdr[0].str.strip()
    df['Case_No'] = hdr[1].str.strip()
    df['Judgment_Date'] = hdr[2].str.replace('Judgment pronounced on', '', regex=False).str.strip()
    df['Facts'] = hdr[3].str.strip().fillna(df['Facts_of_the_Case'])

    df['input_text'] = ('Facts: ' + df['Facts'] + '\nIssues: ' + df['Issues'] +
                        '\nArguments: ' + df['Arguments'] + '\nFinal Order: ' + df['Final_Order'])
    df = df.reset_index(drop=True)
    df.insert(0, 'id', df.index)
    df.to_csv(CLEAN, index=False)
    print('Saved clean.csv')

if not os.path.exists(f'{BASE}/data/processed/test.csv'):
    train, temp = train_test_split(df, test_size=0.2, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)
    for n, p in [('train', train), ('val', val), ('test', test)]:
        p.to_csv(f'{BASE}/data/processed/{n}.csv', index=False)
    print('Saved splits')

print(df.shape)
for c in ['Facts', 'Issues', 'Arguments', 'Ratio_Decidendi', 'Final_Order']:
    w = df[c].str.split().str.len()
    print(f'{c:18s} min {w.min():4d}  median {int(w.median()):4d}  max {w.max():4d} words')
print(os.listdir(f'{BASE}/data/processed'))

Mounted at /content/drive
CSV found: True
Saved clean.csv
Saved splits
(2000, 13)
Facts              min  105  median  129  max  178 words
Issues             min   33  median   51  max   67 words
Arguments          min   66  median  119  max  182 words
Ratio_Decidendi    min   82  median  147  max  219 words
Final_Order        min   30  median   49  max   86 words
['clean.csv', 'train.csv', 'val.csv', 'test.csv']


model part


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


load the modle and test the data


In [ ]:
   from google.colab import drive
   drive.mount('/content/drive')
   !rm -rf /content/drive/MyDrive/judgment-simplifier/models/*
   print('cleaned')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cleaned


In [ ]:
import os, json, torch
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/judgment-simplifier'

!pip -q install transformers accelerate

from transformers import pipeline
test = pd.read_csv(f'{BASE}/data/processed/test.csv')

llm = pipeline('text-generation',
               model='Qwen/Qwen2.5-3B-Instruct',
               dtype=torch.float16, device_map='auto')
print('Model ready, test cases:', len(test))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model ready, test cases: 200


In [ ]:
def summarize(text):
    messages = [
        {'role': 'system', 'content': 'You explain Indian civil court judgments to ordinary people in simple English.'},
        {'role': 'user', 'content': 'Summarize this judgment in 5 short, simple sentences. '
                                    'Say what happened, what the court had to decide, and what it ordered. '
                                    'Use only facts from the text. Avoid legal jargon.\n\n' + text},
    ]
    out = llm(messages, max_new_tokens=250, do_sample=False)
    return out[0]['generated_text'][-1]['content']

row = test.iloc[0]
print('CASE:', row['Case_Name'])
print(summarize(row['input_text']))
print('\nACTUAL ORDER:', row['Final_Order'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


CASE: Meghdoot Agro Industries vs. Padmavathi Gowda and Others


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


The plaintiff asked for a right to use a path near their land. They say they've been using it for many years. But the neighbor says they built a fence there and didn't let them use it. 

The court decided the plaintiff can use a small part of the path. They can walk through it to get to their land. But the neighbor can't block this small part anymore. The court didn't say who would pay for the lawyers' fees.

ACTUAL ORDER: The suit is dismissed in so far as the declaration of a prescriptive easement is concerned. However, having regard to the admitted position that the plaintiff's field is not otherwise approachable by a cart, a limited easement of necessity is declared over the strip shown in red hatching in the Commissioner's map, to the extent of eight feet in width, for the purposes of ingress and egress only. The defendant is restrained from obstructing the said strip. No order as to costs.


In [ ]:
OUT = f'{BASE}/outputs/baseline_summaries.jsonl'
done = set()
if os.path.exists(OUT):
    done = {json.loads(l)['id'] for l in open(OUT)}
print('Already done:', len(done))

with open(OUT, 'a') as f:
    for _, row in test.head(20).iterrows():
        if int(row['id']) in done:
            continue
        s = summarize(row['input_text'])
        f.write(json.dumps({'id': int(row['id']), 'summary': s}) + '\n')
        f.flush()
        print('done', int(row['id']))

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Already done: 0


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 23


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

done 1491


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 480


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 220
done 764


In [ ]:
res = pd.DataFrame([json.loads(l) for l in open(OUT)])
print('Summaries saved:', len(res))

res = res.merge(test[['id', 'Case_Name', 'Final_Order']], on='id')
for _, r in res.head(5).iterrows():
    print('CASE:', r['Case_Name'])
    print('SUMMARY:', r['summary'])
    print('ACTUAL ORDER:', r['Final_Order'])
    print('-' * 80)

Summaries saved: 20
CASE: Meghdoot Agro Industries vs. Padmavathi Gowda and Others
SUMMARY: The plaintiff asked for a right to use a path near their land. They say they've been using it for many years. But the neighbor says they built a fence there and didn't let them use it. 

The court decided the plaintiff can use a small part of the path. They can walk through it to get to their land. But the neighbor can't block this small part anymore. The court didn't say who would pay for the lawyers' fees.
ACTUAL ORDER: The suit is dismissed in so far as the declaration of a prescriptive easement is concerned. However, having regard to the admitted position that the plaintiff's field is not otherwise approachable by a cart, a limited easement of necessity is declared over the strip shown in red hatching in the Commissioner's map, to the extent of eight feet in width, for the purposes of ingress and egress only. The defendant is restrained from obstructing the said strip. No order as to costs.


cell E  extract the ratio for 20 test cases


In [ ]:
def extract_ratio(text):
    messages = [
        {'role': 'system', 'content': 'You are an expert in Indian civil law.'},
        {'role': 'user', 'content': 'Read this judgment. State the ratio decidendi: the legal principle '
                                    'the court applied to decide the case, in 3 to 4 sentences. '
                                    'Mention the relevant legal rule or section, and why it decided the outcome. '
                                    'Do not retell the facts.\n\n' + text},
    ]
    out = llm(messages, max_new_tokens=300, do_sample=False)
    return out[0]['generated_text'][-1]['content']

OUT2 = f'{BASE}/outputs/ratio_predictions.jsonl'
done2 = set()
if os.path.exists(OUT2):
    done2 = {json.loads(l)['id'] for l in open(OUT2)}
print('Already done:', len(done2))

with open(OUT2, 'a') as f:
    for _, row in test.head(20).iterrows():
        if int(row['id']) in done2:
            continue
        r = extract_ratio(row['input_text'])
        f.write(json.dumps({'id': int(row['id']), 'ratio_pred': r}) + '\n')
        f.flush()
        print('done', int(row['id']))

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Already done: 0


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 23


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

done 1491


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 480


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 220
done 764


Cell F.
Score the predicted ratios against the real ratios

In [ ]:
!pip -q install rouge_score bert_score

import os, json
import pandas as pd
from rouge_score import rouge_scorer
from bert_score import score as bert_score

OUT2 = f'{BASE}/outputs/ratio_predictions.jsonl'

pred = pd.DataFrame([json.loads(l) for l in open(OUT2)]).merge(
    test[['id', 'Case_Name', 'Ratio_Decidendi']], on='id')

sc = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
pred['rouge1'] = [sc.score(r, p)['rouge1'].fmeasure for r, p in zip(pred['Ratio_Decidendi'], pred['ratio_pred'])]
pred['rougeL'] = [sc.score(r, p)['rougeL'].fmeasure for r, p in zip(pred['Ratio_Decidendi'], pred['ratio_pred'])]

P, R, F1 = bert_score(pred['ratio_pred'].tolist(), pred['Ratio_Decidendi'].tolist(), lang='en')
pred['bertscore_f1'] = F1.numpy()

print('Cases scored:', len(pred))
print(pred[['rouge1', 'rougeL', 'bertscore_f1']].mean().round(3))

r0 = pred.iloc[0]
print('\nCASE:', r0['Case_Name'])
print('PREDICTED:', r0['ratio_pred'])
print('\nREAL:', r0['Ratio_Decidendi'])

pred.to_csv(f'{BASE}/outputs/ratio_scores.csv', index=False)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Cases scored: 20
rouge1          0.402
rougeL          0.208
bertscore_f1    0.844
dtype: float64

CASE: Meghdoot Agro Industries vs. Padmavathi Gowda and Others
PREDICTED: The ratio decidendi of this judgment is that the court found a limited easement of necessity rather than an easement by prescription. The court held that while the plaintiff had used the track openly and peaceably for over twenty years, the user was not "as of right" but with the permission of the defendant's father, which is inconsistent with an easement by prescription. Therefore, the court did not find an easement by prescription, but instead declared a limited easement of necessity for the purposes of ingress and egress, allowing the plaintiff to use a strip of eight feet in width. This decision was based on the principle that where a person's land is landlocked and there is no reasonable alternative access, an easement of necessity can be created to allow passage through another's land. The court also considere


Cell G: Improved ratio extraction (2 examples)

In [ ]:
train = pd.read_csv(f'{BASE}/data/processed/train.csv')
shots = train.sample(2, random_state=1)

INSTR = ('Read this judgment and state its ratio decidendi: the legal principle and rule the court applied, '
         'and why it decided the outcome. Write 3 to 4 sentences, start directly with the legal principle, '
         'and do not retell the facts.\n\n')

def extract_ratio_v2(text):
    messages = [{'role': 'system', 'content': 'You are an expert in Indian civil law.'}]
    for _, s in shots.iterrows():
        messages.append({'role': 'user', 'content': INSTR + s['input_text']})
        messages.append({'role': 'assistant', 'content': s['Ratio_Decidendi']})
    messages.append({'role': 'user', 'content': INSTR + text})
    out = llm(messages, max_new_tokens=350, do_sample=False)
    return out[0]['generated_text'][-1]['content']

OUT3 = f'{BASE}/outputs/ratio_predictions_v2.jsonl'
done3 = set()
if os.path.exists(OUT3):
    done3 = {json.loads(l)['id'] for l in open(OUT3)}
print('Already done:', len(done3))

with open(OUT3, 'a') as f:
    for _, row in test.head(20).iterrows():
        if int(row['id']) in done3:
            continue
        r = extract_ratio_v2(row['input_text'])
        f.write(json.dumps({'id': int(row['id']), 'ratio_pred': r}) + '\n')
        f.flush()
        print('done', int(row['id']))

[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Already done: 0


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 23


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

done 1491


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 480


[transformers] Both `max_new_tokens` (=350) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 220
done 764


Comparison cell: v1 vs v2 scores

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
sc = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def score_file(path):
    d = pd.DataFrame([json.loads(l) for l in open(path)]).merge(
        test[['id', 'Case_Name', 'Ratio_Decidendi']], on='id')
    r1 = [sc.score(r, p)['rouge1'].fmeasure for r, p in zip(d['Ratio_Decidendi'], d['ratio_pred'])]
    rl = [sc.score(r, p)['rougeL'].fmeasure for r, p in zip(d['Ratio_Decidendi'], d['ratio_pred'])]
    _, _, f1 = bert_score(d['ratio_pred'].tolist(), d['Ratio_Decidendi'].tolist(), lang='en')
    return d, round(sum(r1)/len(r1), 3), round(sum(rl)/len(rl), 3), round(float(f1.mean()), 3)

d2, a, b, c = score_file(OUT3)
print('v1 (plain prompt):   ROUGE-1 0.402  ROUGE-L 0.208  BERTScore 0.844')
print(f'v2 (2 examples):     ROUGE-1 {a}  ROUGE-L {b}  BERTScore {c}')

r = d2.iloc[0]
print('\nCASE:', r['Case_Name'])
print('PREDICTED v2:', r['ratio_pred'])
print('\nREAL:', r['Ratio_Decidendi'])

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


v1 (plain prompt):   ROUGE-1 0.402  ROUGE-L 0.208  BERTScore 0.844
v2 (2 examples):     ROUGE-1 0.397  ROUGE-L 0.229  BERTScore 0.858

CASE: Meghdoot Agro Industries vs. Padmavathi Gowda and Others
PREDICTED v2: An easement by prescription under Section 15 of the Indian Easements Act, 1882, requires continuous and uninterrupted use as of right for twenty years ending within two years prior to the suit. In cases of necessity, however, a shorter period may suffice, and the court can grant an easement if the user is necessary for the enjoyment of the dominant tenement. Here, the plaintiff proved an easement of necessity due to the land's inherent landlocked condition, necessitating an alternative route for ingress and egress.

REAL: An easement of necessity under Section 13 of the Indian Easements Act, 1882 arises only upon severance of a tenement and is confined to what is absolutely necessary for enjoyment of the dominant heritage; mere convenience is not necessity. A prescriptive easem

Cell H: Rewrite the extracted ratio in plain languag

In [ ]:
def simplify_ratio(ratio):
    messages = [
        {'role': 'system', 'content': 'You explain Indian law to ordinary people in simple English.'},
        {'role': 'user', 'content': 'Explain this legal principle in 3 to 4 simple sentences that a Class 8 student '
                                    'could understand. Keep any section numbers and figures exactly as written. '
                                    'Use only what is in the text and add nothing new.\n\n' + ratio},
    ]
    out = llm(messages, max_new_tokens=250, do_sample=False)
    return out[0]['generated_text'][-1]['content']

OUT4 = f'{BASE}/outputs/ratio_simple.jsonl'
done4 = set()
if os.path.exists(OUT4):
    done4 = {json.loads(l)['id'] for l in open(OUT4)}
print('Already done:', len(done4))

with open(OUT4, 'a') as f:
    for _, row in d2.iterrows():
        if int(row['id']) in done4:
            continue
        s = simplify_ratio(row['ratio_pred'])
        f.write(json.dumps({'id': int(row['id']), 'ratio_simple': s}) + '\n')
        f.flush()
        print('done', int(row['id']))

r = d2.iloc[0]
print('\nEXTRACTED RATIO:', r['ratio_pred'])
print('\nPLAIN LANGUAGE:', simplify_ratio(r['ratio_pred']))

[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Already done: 0


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 23


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

done 480


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 220


[transformers] Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done 764

EXTRACTED RATIO: An easement by prescription under Section 15 of the Indian Easements Act, 1882, requires continuous and uninterrupted use as of right for twenty years ending within two years prior to the suit. In cases of necessity, however, a shorter period may suffice, and the court can grant an easement if the user is necessary for the enjoyment of the dominant tenement. Here, the plaintiff proved an easement of necessity due to the land's inherent landlocked condition, necessitating an alternative route for ingress and egress.

PLAIN LANGUAGE: According to the law, if someone uses another person's land without permission for 20 years, they might get a special right to use it. But if they need to use it because their land has no way to get to other places, they can use it even if it's not 20 years. The person who needed to use the land had no other way to reach their property easily, so the court allowed them to use it.
